In [ ]:
model_path = "/content/drive/MyDrive/Capstone_AI/model/final_emotion_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("모델 로드 완료")

In [ ]:
def predict_emotion(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    return id_to_label[pred]

In [ ]:
import os
import anthropic

os.environ["ANTHROPIC_API_KEY"] = "----여기는 api키-----"

client = anthropic.Anthropic()

In [ ]:
def generate_messages(user_text, user_role, selected_mood):
    ai_emotion = predict_emotion(user_text)

    # 사용자용 팝업
    user_prompt = f"""
사용자 역할:
{user_role}

사용자가 직접 선택한 기분:
{selected_mood}

사용자 한 줄 기록:
{user_text}

AI 감정 분석 결과:
{ai_emotion}

사용자 본인에게 보여줄 팝업 메시지를 작성해줘.

조건
- 반드시 1문장
- 30자 이내
- 사용자가 선택한 기분 "{selected_mood}"을 중심으로 공감
- 따뜻한 공감 중심
- 조언 금지
- 이모지 사용 금지
"""

    user_response = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens=80,
        messages=[
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    # 가족용 팝업
    family_prompt = f"""
사용자 역할:
{user_role}

사용자가 직접 선택한 기분:
{selected_mood}

사용자 한 줄 기록:
{user_text}

AI 감정 분석 결과:
{ai_emotion}

가족 구성원에게 보여줄 팝업 알림을 작성해줘.

조건
- 한 문장
- 40자 이내
- "{user_role}" 역할을 자연스럽게 포함
- 일기 원문 그대로 공개 금지
- 감정명 직접 언급 금지
- 사용자가 어떤 상황인지 짧게 요약
- 가족이 건넬 수 있는 자연스러운 말 또는 행동 제안
- 상담사 말투 금지
- 이모지 사용 금지
"""

    family_response = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens=80,
        messages=[
            {
                "role": "user",
                "content": family_prompt
            }
        ]
    )

    return {
        "role": user_role,
        "selected_mood": selected_mood,
        "ai_emotion": ai_emotion,
        "user_message": user_response.content[0].text,
        "family_message": family_response.content[0].text
    }

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # 맨 위로 올리기

!pip install fastapi uvicorn pyngrok

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

ALL_MEMBERS = ["아빠", "엄마", "아들", "딸"]

class MoodRequest(BaseModel):
    mood_text: str
    mood_tag: str
    user_role: str

@app.post("/analyze-mood")
async def analyze_mood(req: MoodRequest):
    family_members = [m for m in ALL_MEMBERS if m != req.user_role]

    result = generate_messages(req.mood_text, req.user_role, req.mood_tag)

    family_messages = {}
    for member in family_members:
        msg = generate_messages(req.mood_text, req.user_role, req.mood_tag)
        family_messages[member] = msg["family_message"]

    return {
        "ai_emotion": result["ai_emotion"],
        "self_message": result["user_message"],
        "family_messages": family_messages
    }

ngrok.set_auth_token("3EYZN55bEzgIVFncnEBtzcFXqC8_4rfLJRdjCnRk5gaz743YL")
public_url = ngrok.connect(8000)
print(f"✅ Flutter에 넣을 URL: {public_url}")

import threading
import asyncio

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000)).serve())

threading.Thread(target=run_server, daemon=True).start()

import time
time.sleep(2)
print("✅ 서버 실행 완료!")